# 08 — Query filters

Query understanding step: extract numeric constraints from the user query with an LLM (instructor), translate them into a Qdrant `query_filter`, keep the original query for the semantic search.

In [19]:
import instructor
import openai
from typing import Literal
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient, models
from dotenv import load_dotenv

load_dotenv("../.env")

qdrant_client = QdrantClient(url="http://localhost:6333")
COLLECTION_NAME = "Recipes-collection-01"

## 1. Constraint extraction schema

In [20]:
class Constraint(BaseModel):
  field: Literal['Calories', 'ProteinContent', 'CarbohydrateContent', 'FatContent', 'total_time_minutes'] = Field(
    description=(
      "Recipe attribute the constraint applies to. Units are fixed: "
      "Calories is kcal; ProteinContent, CarbohydrateContent and FatContent are grams; "
      "total_time_minutes is minutes."
    )
  )
  op: Literal['gt', 'gte', 'lt', 'lte'] = Field(
    description=(
      "Comparison operator. 'more than' -> gt, 'at least' / 'or more' -> gte, "
      "'less than' / 'under' -> lt, 'at most' / 'or less' / 'max' -> lte."
    )
  )
  value: float = Field(
    description=(
      "Threshold, converted to the field's unit when the query uses a different one "
      "(e.g. 'ready in 2 hours' -> 120 for total_time_minutes)."
    )
  )


class ExtractedConstraints(BaseModel):
  reasoning: str = Field(
    description="Short analysis of which numeric thresholds, if any, the query explicitly states."
  )
  constraints: list[Constraint] = Field(
    description="Numeric constraints explicitly stated in the query. Empty if it states none."
  )

## 2. Constraint extractor

In [21]:
extractor_client = instructor.from_provider(
  "openai/gpt-5.4-nano",
  mode=instructor.Mode.RESPONSES_TOOLS,
)

EXTRACTOR_PROMPT = """
You are the query-understanding step of a recipe search pipeline. Given a user query, extract the numeric constraints it explicitly states, so they can be turned into database filters. Semantic search handles everything else about the query; your only job is the numbers.

Rules:
- Extract a constraint ONLY when the query states an explicit numeric threshold. Numbers written as words ("half an hour", "two hours") count as explicit.
- Vague qualifiers ("quick", "light", "low carb", "high protein", "hearty") are NOT constraints. Never invent a number for them.
- Ignore numbers that do not refer to one of the schema fields: servings ("for 4 people"), ingredient counts, oven temperatures, etc.
- A time budget stated by the user ("I have 2 hours") means total_time_minutes <= that value.

Example 1:
query: "find a sweet breakfast with less than 500 cal"
constraints: [{ "field": "Calories", "op": "lt", "value": 500 }]

Example 2:
query: "I have 2 hours and need to make a dinner and I need at least 25g of proteins, what can I do?"
constraints: [{ "field": "total_time_minutes", "op": "lte", "value": 120 }, { "field": "ProteinContent", "op": "gte", "value": 25 }]

Example 3:
query: "quick high protein pasta for 4 people"
constraints: []  (no explicit threshold: "quick" and "high protein" are vague, "4 people" is servings)
"""


def extract_constraints(query: str) -> ExtractedConstraints:
  return extractor_client.create(
    messages=[
      {"role": "system", "content": EXTRACTOR_PROMPT},
      {"role": "user", "content": query},
    ],
    reasoning={"effort": "none"},
    response_model=ExtractedConstraints,
  )

In [22]:
# Each query is paired with the expected constraints for eyeball checking.
TEST_QUERIES = [
  # explicit thresholds
  ("dinner with at least 30 grams of protein", "ProteinContent gte 30"),
  ("quick lunch under 30 minutes", "total_time_minutes lt 30 (and nothing for 'quick')"),
  ("hearty dinner with more than 500 calories", "Calories gt 500"),
  ("low calorie snack, max 150 kcal", "Calories lte 150"),
  # numbers written as words / unit conversion
  ("something I can make in half an hour or less", "total_time_minutes lte 30"),
  ("I have 2 hours, what dinner can I make?", "total_time_minutes lte 120"),
  # traps: must all yield []
  ("I need a low carb lunch", "[]"),
  ("quick high protein breakfast", "[]"),
  ("pasta dinner for 4 people", "[]"),
  ("cookies that bake at 350 degrees", "[]"),
]

for q, expected in TEST_QUERIES:
  r = extract_constraints(q)
  print(q)
  print(f"  expected:    {expected}")
  print(f"  reasoning:   {r.reasoning}")
  print(f"  constraints: {[c.model_dump() for c in r.constraints]}\n")

dinner with at least 30 grams of protein
  expected:    ProteinContent gte 30
  reasoning:   The query explicitly requires at least 30 grams of protein.
  constraints: [{'field': 'ProteinContent', 'op': 'gte', 'value': 30.0}]

quick lunch under 30 minutes
  expected:    total_time_minutes lt 30 (and nothing for 'quick')
  reasoning:   User specifies a time budget: "under 30 minutes". No explicit numeric constraints for calories or macros.
  constraints: [{'field': 'total_time_minutes', 'op': 'lt', 'value': 30.0}]

hearty dinner with more than 500 calories
  expected:    Calories gt 500
  reasoning:   The query explicitly requires meals with more than 500 calories. No other numeric thresholds for protein, carbs, fat, or total time are stated.
  constraints: [{'field': 'Calories', 'op': 'gt', 'value': 500.0}]

low calorie snack, max 150 kcal
  expected:    Calories lte 150
  reasoning:   Query specifies an explicit calorie limit: "max 150 kcal".
  constraints: [{'field': 'Calories', 'op'

## 3. Constraints → Qdrant filter

In [23]:
def get_embedding(text, model="text-embedding-3-small"):
  response = openai.embeddings.create(
    input=text,
    model=model
  )
  return response.data[0].embedding

In [28]:
def build_qdrant_filter(constraints: list[Constraint]) -> models.Filter | None:
  if len(constraints) == 0: return None
  return models.Filter(
    must=[models.FieldCondition(key=c.field, range=models.Range(**{c.op: c.value})) for c in constraints]
  )

In [29]:
def query_points_with_filter(embedding: list[float], qfilter: models.Filter | None, k=5):
  return qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=embedding,
    limit=k,
    query_filter=qfilter,
  )


def query_points_with_fallback(query: str, k=5):
  query_embedding = get_embedding(query)
  extracted = extract_constraints(query)
  qfilter = build_qdrant_filter(extracted.constraints)

  results = query_points_with_filter(query_embedding, qfilter, k)
  filter_relaxed = False
  if qfilter is not None and not results.points:
    results = query_points_with_filter(query_embedding, None, k)
    filter_relaxed = True

  return {
    "results": results,
    "constraints": extracted.constraints,
    "filter_relaxed": filter_relaxed,
  }

In [30]:
def retrieve_with_filters(query: str, k=5) -> dict:
  out = query_points_with_fallback(query, k)
  recipes = []
  for result in out["results"].points:
    payload = result.payload
    recipes.append({
      "id": int(payload["RecipeId"]),
      "name": payload["Name"],
      "score": result.score,
      "rating": payload["bayesian_rating"],
      "n_ratings": payload["n_ratings"],
      "calories": payload.get("Calories"),
      "protein": payload.get("ProteinContent"),
      "carbs": payload.get("CarbohydrateContent"),
      "fat": payload.get("FatContent"),
      "total_time": payload.get("total_time_minutes"),
      "ingredients": payload.get("RecipeIngredientParts") or [],
      "instructions": payload.get("RecipeInstructions") or [],
    })
  return {
    "recipes": recipes,
    "constraints": [c.model_dump() for c in out["constraints"]],
    "filter_relaxed": out["filter_relaxed"],
  }

In [31]:
retrieve_with_filters("I have less than one 1 hour to get a proper dinner for my family, what can I do?")

{'recipes': [{'id': 206781,
   'name': 'Last Minute Lasagna',
   'score': 0.4870426,
   'rating': 4.692244584372267,
   'n_ratings': 1,
   'calories': 436.3,
   'protein': 24.4,
   'carbs': 28.7,
   'fat': 26.0,
   'total_time': 50,
   'ingredients': ['frozen spinach', 'mozzarella cheese', 'parmesan cheese'],
   'instructions': ['Heat oven to 375 degrees.',
    'Spoon a thin layer of sauce over bottom of a 9x13 baking dish.',
    'Cover with a single layer of ravioli.',
    'Top with half the spinach, half the olives, half the mozzarella, and a third of the sauce.',
    'Repeat with another layer of ravioli, the remaining spinach, olives and mozzarella, and half the remaining sauce.',
    'Top with a final layer of ravioli and the remaining sauce (not all of the ravioli may be needed).',
    'Sprinkle with the parmesan cheese. I also topped it with more mozzarella.',
    'Cover with foil and bake for 30 minutes. (I baked 45 minutes to compensate for the frozen ravioli).',
    'Uncover 

In [32]:
# Sanity checks: the first must return only recipes under 300 kcal,
# the second must trigger the fallback (filter_relaxed=True).
for q in ["breakfast under 300 calories", "dinner under 5 calories"]:
  out = retrieve_with_filters(q)
  print(q)
  print(f"  constraints:    {out['constraints']}")
  print(f"  filter_relaxed: {out['filter_relaxed']}")
  for r in out["recipes"]:
    print(f"    {r['name']} — {r['calories']} kcal, {r['total_time']} min")
  print()

breakfast under 300 calories
  constraints:    [{'field': 'Calories', 'op': 'lt', 'value': 300.0}]
  filter_relaxed: False
    Extreme Low-Fat Buttermilk-Bran Breakfast Squares — 160.3 kcal, 70 min
    Low Fat Egg McMuffin — 140.7 kcal, 17 min
    Peach Melba Breakfast — 172.0 kcal, 5 min
    Quick and Easy Fibre Breakfast — 231.0 kcal, 5 min
    Portabella and Spinach Eggs Benedict — 163.9 kcal, 10 min

dinner under 5 calories
  constraints:    [{'field': 'Calories', 'op': 'lt', 'value': 5.0}]
  filter_relaxed: False
    Light Chicken Caprese — 0.0 kcal, 25 min
    Veggie Crescents — 0.0 kcal, 20 min
    Country Soup — 0.0 kcal, 45 min
    Simple Spaghetti — 0.0 kcal, 2 min
    Perfect No Burn Popcorn - Every Kernal Pops and It's Healthy Too — 0.0 kcal, 7 min

